# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² colorectal cancer dataset using the `mlcroissant` library, referencing all schema, fields, and record sets by their `@id` fields for transparency and reproducibility.

### Dataset Source
The dataset schema is accessible via the following Croissant JSON-LD URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Install mlcroissant in your environment if not already installed
!pip install mlcroissant

## 1. Data Loading

We load the FAIR² dataset using `mlcroissant`, and display basic summary metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the URL to the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset package (schema and metadata only)
dataset = mlc.Dataset(croissant_url)

# Print dataset-level metadata summary
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Citation: {dataset.metadata.citeAs}")

## 2. Data Overview

We review the available record sets and their fields, referencing by `@id`. This guides us in selecting the appropriate elements for extracting and analyzing data.

Note: The list of record sets is determined by inspecting the dataset schema. We'll enumerate all available record sets and their key fields with their respective `@id`s.

In [ ]:
# List all record sets, their `@id`, and their available fields by `@id`
for record_set in dataset.record_sets:
    print(f"Record Set: {record_set['@id']}")
    if 'field' in record_set:
        fields = record_set['field'] if isinstance(record_set['field'], list) else [record_set['field']]
        for field in fields:
            if isinstance(field, dict):
                print(f"  └─ Field: {field['@id']} (label: {field.get('rdfs:label', field.get('schema:name',''))})")
            else:
                print(f"  └─ Field: {field}")
    else:
        print("  └─ No explicit fields defined.")
    print()

## 3. Data Extraction

We will use the primary clinical records record set to extract tabular data. All extractions will refer to record sets and field names by their `@id` as per best practice.

Let's identify the main clinical record set by checking the previous cell output. For demonstration purposes, we will proceed using the first tabular (data) record set found.

In [ ]:
# Choose main record set by its @id; if unsure, use the first tabular set found

# Find the first record set that can be loaded as tabular data
tabular_record_sets = [rs for rs in dataset.record_sets if 'field' in rs]
record_set_ids = [rs['@id'] for rs in tabular_record_sets]

# Load all available tabular record sets into pandas DataFrames
dataframes = {}
for rs_id in record_set_ids:
    print(f"Extracting data for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns: {list(df.columns)}")
    print(df.head(2))
    print('-'*40)

# We'll proceed with the main clinical records record set for further analysis
main_record_set_id = record_set_ids[0]  # Change if primary clinical table has a different @id
print(f"Main analysis will use record set: {main_record_set_id}")
print(f"Field @ids: {list(dataframes[main_record_set_id].columns)}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

In this section, we process and analyze the dataset. We:
- Select a numerical field (referenced by its `@id`).
- Filter records based on a threshold value.
- Normalize the numeric variable.
- Group records by a categorical field (also referenced by its `@id`).

Inspect the DataFrame columns above to determine available numeric and categorical fields. We'll fill in the field `@id`s found in the dataset.

In [ ]:
# Select field @id's appropriate for numeric and categorical field analysis
df = dataframes[main_record_set_id]

# Heuristically (or manually) pick a numeric field
numeric_candidates = [col for col in df.columns if df[col].dtype in ['float64','int64']]
if len(numeric_candidates) == 0:
    # Try to coerce some columns to numeric
    for col in df.columns:
        coerced = pd.to_numeric(df[col], errors='coerce')
        if coerced.notna().sum() > 0 and coerced.notna().sum() >= (0.7 * len(df)):
            numeric_candidates.append(col)

# Pick first numeric candidate for demonstration
numeric_field_id = numeric_candidates[0] if numeric_candidates else df.columns[0]

print(f"Using numeric field: {numeric_field_id}")
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Define a threshold for filtering
threshold = df[numeric_field_id].median() if pd.notnull(df[numeric_field_id].median()) else 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Records with '{numeric_field_id}' > {threshold}")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Find a categorical (group) field
categorical_candidates = [col for col in df.columns if df[col].dtype == 'object' and \
                          df[col].nunique() > 1 and df[col].nunique() <= 10]
group_field = categorical_candidates[0] if categorical_candidates else df.columns[1]
print(f"\nGrouping by field: {group_field}")

if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame(name=f"{numeric_field_id}_mean")
    print(f"\nMean of '{numeric_field_id}' grouped by '{group_field}':")
    print(grouped_df.head())

## 5. Visualization

Now, let's visualize data distributions and relationships between numeric and categorical fields, all referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (referenced by @id)
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20, color='dodgerblue')
plt.title(f"Distribution of '{numeric_field_id}'")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot by categorical field if reasonable groupings found
if group_field in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=df)
    plt.title(f"'{numeric_field_id}' by '{group_field}'")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

We've loaded and explored the FAIR² clinical dataset using the `mlcroissant` library and Python tools:
- All data elements (record sets, fields) were referenced by their `@id` as required by the Croissant standard.
- We displayed, filtered, normalized, grouped, and visualized clinical records on field-level metadata.

This notebook serves as a template for deeper clinical, epidemiological, or data science investigations using FAIR-aligned health research datasets.

<sup>Notebook generated using the mlcroissant API and dataset schema from: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)</sup>